In [129]:
import numpy as np
import pandas as pd

here also we should do text preprocessing part, because predict from user's input

In [130]:
import re
import string

In [131]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [132]:
#open the downloaded english stop words file
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines() #take line by line as a list

In [133]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [134]:
from nltk.stem import SnowballStemmer
snowball = SnowballStemmer("english")


In [135]:
#create a function includeing all steps in preprocessing
def preprocessing(text):
    data = pd.DataFrame([text], columns=["tweet"])
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x,flags=re.MULTILINE) for x in x.split()))
    data["tweet"] = data["tweet"].apply(remove_punctuations)
    data["tweet"] = data["tweet"].str.replace(r'\d+', '', regex=True)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in ['a','an','the','and','or','but','if','not','then']))
    data["tweet"] = data["tweet"].apply(lambda x: " ". join(x for x in x.split() if x not in sw))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(snowball.stem(x) for x in x.split()))
    #data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]

get the downloaded vocabulary

In [136]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
tokens = vocab[0].tolist()

In [137]:
tokens

['fingerprint',
 'pregnanc',
 'test',
 'android',
 'app',
 'beauti',
 'cute',
 'health',
 'iger',
 'iphoneon',
 'iphonesia',
 'iphon',
 'final',
 'transpar',
 'silicon',
 'case',
 'thank',
 'uncl',
 'yay',
 'soni',
 'xperia',
 'sonyexperias…',
 'love',
 'would',
 'go',
 'talk',
 'makememori',
 'unplug',
 'relax',
 'smartphon',
 'wifi',
 'connect',
 'im',
 'wire',
 'know',
 'georg',
 'made',
 'way',
 'daventri',
 'home',
 'amaz',
 'servic',
 'appl',
 'wont',
 'even',
 'question',
 'unless',
 'pay',
 'stupid',
 'support',
 'softwar',
 'updat',
 'fuck',
 'phone',
 'big',
 'time',
 'happi',
 'us',
 'instap',
 'instadaili',
 'xperiaz',
 'new',
 'type',
 'c',
 'charger',
 'cabl',
 'uk',
 '…',
 'bay',
 'amazon',
 'etsi',
 'year',
 'rob',
 'cross',
 'tobi',
 'young',
 'evemun',
 'mcmafia',
 'taylor',
 'spectr',
 'newyear',
 'start',
 'recip',
 'technolog',
 'samsunggalaxi',
 'iphonex',
 'pictwittercompjiwqwtc',
 'bout',
 'shop',
 'listen',
 'music',
 'justm',
 'likeforlik',
 'followforfollow…'

vectorization

In [138]:
def vectorizer(ds, vocabulary):
    # Create an empty list to store the vectorized sentences
    vectorized_list = []

    # Loop through each sentence in the dataset
    for sentence in ds:
        # Create a vector of zeros with the same length as the vocabulary
        # Each position represents a word in the vocabulary
        sentence_list = np.zeros(len(vocabulary))

        # Loop through each word in the vocabulary
        for i in range(len(vocabulary)):

            # Check if the vocabulary word exists in the sentence
            # sentence.split() converts the sentence into a list of words
            if vocabulary[i] in sentence.split():

                # If the word is present, set the corresponding position to 1
                # This represents the presence of that word in the sentence
                sentence_list[i] = 1

        # After checking all vocabulary words, add the vector to the list
        vectorized_list.append(sentence_list)

    # Convert the list of vectors into a NumPy array with float32 datatype
    vectorized_list_new = np.asarray(vectorized_list, dtype=np.float32)

    # Return the vectorized dataset
    return vectorized_list_new

import model for predictions

In [139]:
import pickle

In [140]:
#open the model
with open('../static/model/model.pickle', 'rb') as f:
    model = pickle.load(f)

In [141]:
with open('../static/model/tfidf_vectorizer.pickle', 'rb') as f:
    tfidf = pickle.load(f)

In [142]:
#take as positive or negative
def get_prediction(text):
    processed_series = preprocessing(text)
    processed_str = processed_series.iloc[0]
    
    vector = tfidf.transform([processed_str])
    
    proba_negative = model.predict_proba(vector)[0][1]
    threshold = 0.68          # Change to 0.60 or 0.68 if needed
    
    return "negative" if proba_negative >= threshold else "positive"

In [149]:
def debug_vectorization(text, vocab_list):
    # Get the processed result (it's a Series)
    processed_series = preprocessing(text)
    
    # Extract the actual string
    processed_str = processed_series.iloc[0]          # or processed_series[0]
    
    words = processed_str.split()
    vocab_set = set(vocab_list)                       # convert to set for fast lookup
    
    known = [w for w in words if w in vocab_set]
    unknown = [w for w in words if w not in vocab_set]
    
    print("Original text     :", text)
    print("After preprocessing:", processed_str)
    print("Words in sentence :", words)
    print("Known in vocab    :", known)
    print("Unknown / lost    :", unknown)
    print(f"Vocabulary coverage: {len(known)/max(1, len(words)):.1%}")
    print("-" * 60)

create a text for test

In [150]:
txt = 'not bad'

In [151]:
debug_vectorization(txt, tokens)

Original text     : not bad
After preprocessing: bad
Words in sentence : ['bad']
Known in vocab    : ['bad']
Unknown / lost    : []
Vocabulary coverage: 100.0%
------------------------------------------------------------


In [152]:
#preprocess the txt
preprocessed_txt = preprocessing(txt)

In [153]:
preprocessed_txt

0    bad
Name: tweet, dtype: str

In [154]:
vectorized_txt = vectorizer(preprocessed_txt, tokens)

In [155]:
vectorized_txt

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 15925), dtype=float32)

In [156]:
print("Number of words that became 1 in vector:", vectorized_txt.sum())
print("Total features in vocab:", vectorized_txt.shape[1])

Number of words that became 1 in vector: 1.0
Total features in vocab: 15925


In [159]:
prediction = get_prediction(txt)
print(prediction)

ValueError: X has 15925 features, but LogisticRegression is expecting 9292 features as input.

another example

In [128]:
test_sentences = [
    "Is this the way Daraz send us products ???? 😡 S***, worst experience?? WE ARE REGULAR BUYERS FROM DARAZ AND WE ARE NOT PLUCKING MONEY FROM TREES, DELIVERY IS REALLY BAAD, anyway I don't think this is a fault of the store. since seller did it job ILL NOT RETURN THIS A** DARAZ PLEASE CARE ABOUT CUSTOMERS, SEEMS YOUR WORKERS DONT HAVE DECSIPLINE BRUHHHHH............If this happens CONTINUOUSLY we have stop buying from THIS !!!",
    "Good but sometimes changed the weight when i measure multiple times but great",
    "this is okay for daily use",
    "Excellent product . I got gift with this product thank you  ",
    "this product is pure shit dont waste your money",
    "it's a good product, cleansing my face so good and it actually helped me to have a clear skin , fast delivery and well packed",
    "horrible dont recommend to anyone"
]

for sentence in test_sentences:
    print(f"Text: {sentence}")
    print(f"Prediction: {get_prediction(sentence)}")
    print("-" * 50)

Text: Is this the way Daraz send us products ???? 😡 S***, worst experience?? WE ARE REGULAR BUYERS FROM DARAZ AND WE ARE NOT PLUCKING MONEY FROM TREES, DELIVERY IS REALLY BAAD, anyway I don't think this is a fault of the store. since seller did it job ILL NOT RETURN THIS A** DARAZ PLEASE CARE ABOUT CUSTOMERS, SEEMS YOUR WORKERS DONT HAVE DECSIPLINE BRUHHHHH............If this happens CONTINUOUSLY we have stop buying from THIS !!!
Prediction: negative
--------------------------------------------------
Text: Good but sometimes changed the weight when i measure multiple times but great
Prediction: positive
--------------------------------------------------
Text: this is okay for daily use
Prediction: positive
--------------------------------------------------
Text: Excellent product . I got gift with this product thank you  
Prediction: positive
--------------------------------------------------
Text: this product is pure shit dont waste your money
Prediction: negative
-------------------